**Menghubungkan Google Drive**

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Install library
!pip install transformers datasets torch tensorflow gensim pyLDAvis Sastrawi nltk emoji

# **Import Library**

In [ ]:
# Import library
import pandas as pd
import numpy as np
import re
import emoji
from Sastrawi.StopWordRemover.StopWordRemoverFactory import StopWordRemoverFactory
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm
import string
from sklearn.feature_extraction.text import CountVectorizer
from gensim import corpora, models
import gensim
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from gensim import corpora
from gensim.models import LdaModel
from gensim.models import CoherenceModel

# **Load Dataset**

In [ ]:
# Load the dataset
mbg = pd.read_csv('/content/drive/MyDrive/Tugas Akhir/Data/datafix.csv', sep=';', encoding='latin-1')
mbg

In [ ]:
# Check NA or missing values
mbg = mbg.dropna(subset=['text']).reset_index(drop=True)
mbg

In [ ]:
# Hapus kolom kecuali text
mbg.drop(columns=[
    'videoWebUrl', 'submittedVideoUrl', 'input', 'cid', 'createTime',
       'createTimeISO','diggCount', 'likedByAuthor', 'pinnedByAuthor',
       'repliesToId', 'replyCommentTotal', 'uid', 'uniqueId',
       'avatarThumbnail', 'mentions', 'detailedMentions', 'Unnamed: 17',
       'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21',
       'Unnamed: 22'
], inplace=True, errors='ignore')

mbg.head()

In [ ]:
mbg = pd.read_csv('/content/drive/MyDrive/Tugas Akhir/Data/mbgtanpaNAbaru.csv', sep=';', encoding='latin-1')
mbg

# **Pre-Processing**

In [ ]:
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')
nltk.download('stopwords')
nltk.download('punkt_tab')

# Load Stopword dan Stemmer

stop_words = set(stopwords.words('indonesian'))
tambahan = {'deh', 'dong', 'sih', 'lah', 'nih', 'tuh', 'nya', 'mua', 'nyh'}
stop_words = stop_words.union(tambahan)

factory = StemmerFactory()
stemmer = factory.create_stemmer()

In [ ]:
# Load Kamus Normalisasi

df_norm = pd.read_csv(
    '/content/drive/MyDrive/Tugas Akhir/Data/normalbaru.txt',
    sep=',',
    header=None,
    names=["salah","benar"],
    engine='python',
    on_bad_lines='skip'
)

In [ ]:
df_norm = df_norm.dropna()

In [ ]:
df_norm.head(5)

In [ ]:
df_norm.tail(5)

In [ ]:
kamus_norm = dict(zip(df_norm['salah'], df_norm['benar']))

In [ ]:
with open('/content/drive/MyDrive/Tugas Akhir/Data/normalbaru.txt', 'r') as f:
    for i, line in enumerate(f):
        if line.count(',') != 1:
            print(f"Baris {i}: {line}")

In [ ]:
# cek apakah ada NaN di kamus
print(any(pd.isnull(v) for v in kamus_norm.values()))

In [ ]:
# Case Folding

def case_folding(text):
    if pd.isnull(text):
        return ""
    return text.lower()

mbg['case_folding'] = mbg['text'].apply(case_folding)


In [ ]:
# Data Cleansing

def cleansing(text):

    text = emoji.replace_emoji(text, "")

    text = re.sub(r"http\S+|www\S+", "", text)

    text = re.sub(r"@\w+|#\w+", "", text)

    text = re.sub(r'(\d+)[\.,](\d+)', r'\1DECIMAL\2', text) # Melindungi desimal

    text = re.sub(r"[^a-zA-Z0-9\s]", " ", text) # Hapus Tanda baca

    text = re.sub(r"\s+", " ", text).strip()

    text = re.sub(r'(.)\1{2,}', r'\1', text)

    text = re.sub(r'(\d+)DECIMAL(\d+)', r'\1.\2', text) # balikin ke desimal

    # Hapus punctuation berlebih
    text = re.sub(r'[^\w\s]', ' ', text)

    # Hapus multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()

    # hapus variasi wkwk (lebih fleksibel)
    text = re.sub(r'\bwk+\b', '', text)

    # hapus variasi haha
    text = re.sub(r'\bha(ha)+\b', '', text)

    # hapus variasi hmm
    text = re.sub(r'\bhm+\b', '', text)

    # hapus variasi p pp
    text = re.sub(r'\bp+\b', '', text)
    return text

mbg['clean_text'] = mbg['case_folding'].apply(cleansing)

# hapus data kosong
mbg = mbg[mbg['clean_text'].str.strip() != ""]

In [ ]:
# Cek NA di Clean_Text
mbg['clean_text'].isnull().sum()

In [ ]:
mbg

In [ ]:
mbg = mbg[mbg['clean_text'].str.strip() != ""].copy()

In [ ]:
# Normalisasi
def normalisasi(text):

    text = str(text)

    words = text.split()

    normalized = []
    for word in words:
        hasil = kamus_norm.get(word, word)

        if pd.isnull(hasil) or hasil == '':
            hasil = word

        normalized.append(str(hasil))

    return " ".join(normalized)

mbg = mbg.copy()
mbg['normalized_text'] = mbg['clean_text'].apply(normalisasi)

In [ ]:
mbg

In [ ]:
# Tokenizing

def tokenizing(text):

    tokens = word_tokenize(text)

    return tokens

mbg['tokens'] = mbg['normalized_text'].apply(tokenizing)

In [ ]:
# Remove Stopwords

def remove_stopwords(tokens):

    filtered = [word for word in tokens if word not in stop_words]

    return filtered

mbg['stopword_removal'] = mbg['tokens'].apply(remove_stopwords)

In [ ]:
# Stemming

def stemming(tokens):

    stemmed = [stemmer.stem(word) for word in tokens]

    return stemmed

mbg['tokens_stemmed'] = mbg['stopword_removal'].apply(stemming)

In [ ]:
mbg.columns

In [ ]:
# Simpan data
mbg.to_csv('/content/drive/MyDrive/Tugas Akhir/Data/mbgdatabersihya.csv', index=False)

In [ ]:
mbg

In [ ]:
mbg.head(5)

In [ ]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

mbg.head(5)

In [ ]:
# Simpan hasil preprocessing
mbg.to_csv('/content/drive/MyDrive/Tugas Akhir/Data/mbghasilpreprosesya3.csv', index=False)

In [ ]:
# Simpan hanya kolom text, clean_text, dan normalized_text
mbgsimpan = mbg[['normalized_text']]
mbgsimpan

In [ ]:
mbgsimpan.to_csv('/content/drive/MyDrive/Tugas Akhir/Data/kolomnormalisasi.csv', index=False)

In [ ]:
mbg.shape

In [ ]:
# Check NA
mbg.isnull().sum()

In [ ]:
# Load Dataset After Pre-processing
mbg_baru = pd.read_csv('/content/drive/MyDrive/Tugas Akhir/Data/mbghasilpreprosesya3.csv')
mbg_baru

In [ ]:
import pandas as pd

pd.set_option('display.max_colwidth', None)

mbg_baru.head(5)

In [ ]:
mbg_baru.tail(5)

In [ ]:
mbg_baru.columns

# **Aspect Exctraction using LDA**

In [ ]:
from gensim import corpora
import ast

# Convert string representation of lists in 'tokens_lda' to actual lists
texts = mbg_baru['tokens_stemmed'].apply(ast.literal_eval)

id2word = corpora.Dictionary(texts) # membuat dictionary
id2word.filter_extremes(no_below=5, no_above=0.5)  # filter kata yang terlalu jarang atau terlalu sering (disesuaikan)

corpus = [id2word.doc2bow(text) for text in texts] # mengubah teks menjadi representasi numerik (bag-of-words/BOW)

In [ ]:
# lihat 10 kata pertama beserta id-nya
for i, (id, word) in enumerate(id2word.iteritems()):
    if i < 10:
        print(f"ID: {id}, Word: {word}")

In [ ]:
print(id2word.token2id)  # dictionary: word → id


In [ ]:
# Dokumen 0
print(corpus[0])

In [ ]:
doc = corpus[0]

for word_id, freq in doc:
    print(f"Kata: {id2word[word_id]}, Frekuensi: {freq}")

In [ ]:
for i in range(5):
    print(f"\nDokumen {i}:")
    for word_id, freq in corpus[i]:
        print(f"{id2word[word_id]}: {freq}")

In [ ]:
import pandas as pd

doc = corpus[0]

data = [(id2word[word_id], word_id, freq) for word_id, freq in doc]

df_bow = pd.DataFrame(data, columns=['word', 'word_id', 'frequency'])
print(df_bow)

In [ ]:
# Total dokumen sama dengan banyaknya data yang ada
len(corpus)

In [ ]:
#  Import Library
import gensim
from gensim import corpora
from gensim.models import CoherenceModel
import matplotlib.pyplot as plt
import random
import numpy as np

# Set Seed untuk Reproducibility
random.seed(100)
np.random.seed(100)

In [ ]:
# Fungsi Hitung Coherence
def compute_coherence(dictionary, corpus, texts, start=2, limit=16, step=1):
    model_list = []
    coherence_values = []

    for k in range(start, limit + 1, step):
        model = gensim.models.LdaModel(
            corpus=corpus,
            id2word=dictionary,
            num_topics=k,
            random_state=100,
            chunksize=200,
            passes=10,
            alpha='auto',
            per_word_topics=True
        )
        model_list.append(model)

        coherence_model = CoherenceModel(
            model=model,
            texts=texts,
            dictionary=dictionary,
            coherence='c_v'
        )
        coherence_values.append(coherence_model.get_coherence())

    return model_list, coherence_values

# Jalankan Fungsi
model_list, coherence_values = compute_coherence(
    dictionary=id2word,
    corpus=corpus,
    texts=texts,
    start=2,
    limit=16,
    step=1
)

# Print Nilai Coherence Tiap k
for i, cv in enumerate(coherence_values, start=2):
    print(f"Jumlah topik (k) = {i}, Coherence Score = {cv:.4f}")



In [ ]:
# Visualisasi Coherence
x = range(2, 17)

plt.plot(x, coherence_values)
plt.xlabel("Jumlah Topik (k)")
plt.ylabel("Coherence Score")
plt.title("Coherence Score vs Jumlah Topik")
plt.show()

In [ ]:
# Pilih k Terbaik
#  k terbaik berdasarkan grafik atau dari nilai tertinggi
optimal_k = x[np.argmax(coherence_values)]
print(f"\nJumlah topik terbaik berdasarkan coherence score adalah: {optimal_k}")

In [ ]:
# Buat Model LDA Final
lda_model_final = gensim.models.LdaModel(
    corpus=corpus,
    id2word=id2word,
    num_topics=optimal_k,
    random_state=100,
    chunksize=200,
    passes=20,
    alpha='auto',
    per_word_topics=True
)
lda_model_final

In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Visualisasi Topik dengan pyLDAvis
import pyLDAvis
import pyLDAvis.gensim_models

pyLDAvis.enable_notebook()  # kalau di Jupyter Notebook
vis = pyLDAvis.gensim_models.prepare(lda_model_final, corpus, id2word)
pyLDAvis.display(vis)

Overall term frequency : Frekuensi istilah secara keseluruhan -> biru

Estimated term frequency within the selected topic : Perkiraan frekuensi istilah dalam topik yang dipilih -> merah

In [ ]:
# Kata Kunci Tiap Topik (Top Words per Topic)
for idx, topic in lda_model_final.show_topics(formatted=False, num_words=10):
    print(f"\nTopik {idx}:")
    print([word for word, _ in topic])

In [ ]:
for idx, topic in lda_model_final.show_topics(formatted=False, num_words=10):
    print(f"\nTopik {idx}:")
    for word, weight in topic:
        print(f"{word}: {weight:.4f}")

In [ ]:
# Mendapatkan topik dominan + persentase kontribusi + keywords
def get_dominant_topic(ldamodel, corpus, texts):
    topics_per_doc = []
    perc_contrib = []
    topic_keywords = []

    for i, row in enumerate(ldamodel[corpus]):
        row = sorted(row[0], key=lambda x: (x[1]), reverse=True)

        # ambil topik dominan
        topic_num, prop_topic = row[0]

        # ambil keywords topik tersebut
        keywords = ldamodel.show_topic(topic_num, topn=10)
        keywords = ", ".join([word for word, _ in keywords])

        topics_per_doc.append(topic_num)
        perc_contrib.append(prop_topic)
        topic_keywords.append(keywords)

    return topics_per_doc, perc_contrib, topic_keywords

dominant_topics, perc_contribution, topic_keywords = get_dominant_topic(
    lda_model_final, corpus, texts
)

In [ ]:
mbg_baru['dominant_topic'] = dominant_topics
mbg_baru['perc_contrib'] = perc_contribution
mbg_baru['topic_keywords'] = topic_keywords
mbg_baru

In [ ]:
topic_names = {
    0: "Ketidaksetujuan & Risiko Program",
    1: "Pelaksanaan Program di Sekolah",
    2: "Dukungan terhadap Program"
}

mbg_baru['aspect'] = mbg_baru['dominant_topic'].map(topic_names)
mbg_baru

In [ ]:
mbg_baru.columns

In [ ]:
mbg_baru.dtypes

In [ ]:
# Simpan data setelah preprocessing
mbg_baru.to_csv('/content/drive/MyDrive/Tugas Akhir/Data/aspekmbgada3barubgty.csv', index = False)

In [ ]:
# Hitung jumlah masing-masing aspek
aspect_count= mbg_baru['aspect'].value_counts().sort_index()
print(aspect_count)

In [ ]:
# Visualisasi
aspect_count.plot(kind='bar')

In [ ]:
# Visualisasi Pie-chart beserta persentasenya
aspect_count.plot(kind='pie', autopct='%1.1f%%')


In [ ]:
# Load Dataset After Pre-processing & Ekstraksi Aspek
mbg_baru = pd.read_csv('/content/drive/MyDrive/Tugas Akhir/Data/aspekmbgada3barubgty.csv')
mbg_baru

In [ ]:
mbg_baru.columns

In [ ]:
mbg_baru_norspe = mbg_baru[['normalized_text', 'aspect']]
mbg_baru_norspe

In [ ]:
# Simpan csv
mbg_baru_norspe.to_csv('/content/drive/MyDrive/Tugas Akhir/Data/hanyanormalaspek.csv', index = False)

# **Opinion Extraction**

## **Using IndoBERT**
model pre-trained seperti mdhugol/indonesia-bert-sentiment-classification

In [ ]:
mbg_baru = pd.read_csv('/content/drive/MyDrive/Tugas Akhir/Data/normalisasidanaspek.csv', sep = ";")
mbg_baru

In [ ]:
# Check missing value
mbg_baru.isnull().sum()

In [ ]:
#  Load IndoBERT Sentiment
from transformers import pipeline
from transformers import AutoTokenizer, AutoModelForSequenceClassification

pretrained= "mdhugol/indonesia-bert-sentiment-classification"

model = AutoModelForSequenceClassification.from_pretrained(pretrained)
tokenizer = AutoTokenizer.from_pretrained(pretrained)

sentiment_analysis = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

label_index = {'LABEL_0': 'positive', 'LABEL_1': 'neutral', 'LABEL_2': 'negative'}


In [ ]:
# Contoh
pos_text = "Sangat bahagia hari ini"
neg_text = "Dasar anak sialan!! Kurang ajar!!"

result = sentiment_analysis(pos_text)
status = label_index[result[0]['label']]
score = result[0]['score']
print(f'Text: {pos_text} | Label : {status} ({score * 100:.3f}%)')

result = sentiment_analysis(neg_text)
status = label_index[result[0]['label']]
score = result[0]['score']
print(f'Text: {neg_text} | Label : {status} ({score * 100:.3f}%)')

In [ ]:
import pandas as pd

# fungsi untuk ambil label & score
def get_sentiment(text):
    try:
        result = sentiment_analysis(text)[0]
        label = label_index[result['label']]
        score = result['score']
        return pd.Series([label, score])
    except:
        return pd.Series([None, None])

# apply ke kolom normalized_text
mbg_baru[['sentiment', 'score']] = mbg_baru['normalized_text'].apply(get_sentiment)

In [ ]:
mbg_baru

In [ ]:
mbg_baru["sentiment"].unique()

In [ ]:
# Mengubah Sentiment Label menjadi Int
label2id = {"positive": 0, "neutral": 1, "negative": 2}
mbg_baru["sentiment_int"] = mbg_baru["sentiment"].map(label2id)

In [ ]:
mbg_baru

In [ ]:
# Simpan csv
mbg_baru.to_csv('/content/drive/MyDrive/Tugas Akhir/Data/pelabelansentimenbaruya.csv', index = False)

In [ ]:
mbg_baru

In [ ]:
# Hitung jumlah masing-masing sentimen untuk tiap aspek
sentiment_count_indb = mbg_baru.groupby(['aspect', 'sentiment_int_indobert']).size().unstack(fill_value=0)

# Tampilkan hasil
print(sentiment_count_indb)

In [ ]:
# Hitung jumlah masing-masing sentimen untuk tiap aspek
sentiment_count = mbg_baru.groupby(['aspect', 'sentiment_int']).size().unstack(fill_value=0)

# Tampilkan hasil
print(sentiment_count)

In [ ]:
# Hitung jumlah masing-masing sentimen
sentiment_count_indb = mbg_baru['sentiment_int_indobert'].value_counts().sort_index()
print(sentiment_count_indb)

In [ ]:
# Hitung jumlah masing-masing sentimen
sentiment_count = mbg_baru['sentiment_int'].value_counts().sort_index()
print(sentiment_count)

In [ ]:
# Hitung jumlah masing-masing sentimen
sentiment_counts= mbg_baru.groupby(['sentiment']).size()

# Tampilkan hasil
print(sentiment_counts)

In [ ]:
# Pie Chart
sentiment_counts.plot(kind='pie', autopct='%1.1f%%')

In [ ]:
# Pie Chart
sentiment_count_indb.plot(kind='pie', autopct='%1.1f%%')

In [ ]:
avg_sentiment = mbg_baru.groupby('aspect')['sentiment_int'].mean()
print(avg_sentiment)

In [ ]:
import matplotlib.pyplot as plt

avg_sentiment.plot(kind='bar', figsize=(8,5))

plt.title('Rata-rata Sentimen per Aspek')
plt.xlabel('Aspek')
plt.ylabel('Skor Sentimen')
plt.axhline(0)  # garis netral

plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
sentiment_dan_aspect = mbg_baru.groupby(['aspect', 'sentiment']).size().unstack(fill_value=0)
print(sentiment_dan_aspect)

In [ ]:
import matplotlib.pyplot as plt

sentiment_dan_aspect.plot(
    kind='bar',
    stacked=True,
    figsize=(10,6)
)

plt.title('Distribusi Sentimen Berdasarkan Aspek')
plt.xlabel('Aspek')
plt.ylabel('Jumlah')
plt.xticks(rotation=0)
plt.legend(title='Sentimen')

plt.tight_layout()
plt.show()

# New Section

# **Runtime Keputus, jadi langsung pake data terakhir yang disimpan (udah bersih, ada aspek, dan sentimennya**)

In [ ]:
mbg_baru = pd.read_csv('/content/drive/MyDrive/Tugas Akhir/Data/pelabelansentimenbaruya.csv')
mbg_baru

In [ ]:
# Cek missing values
mbg_baru.isnull().sum()

In [ ]:
# Cek tipe data
mbg_baru.dtypes

In [ ]:
# Melihat total sentimen berdasarkan aspek
# Hitung jumlah sentimen per aspek
sentimen_count = mbg_baru.groupby(['aspect', 'sentiment']).size().reset_index(name='jumlah')
print(sentimen_count)

# Atau tampilkan dalam bentuk tabel pivot yang lebih mudah dibaca
sentimen_pivot = mbg_baru.groupby(['aspect', 'sentiment']).size().unstack(fill_value=0)
print(sentimen_pivot)

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Hitung frekuensi
sentimen_count = mbg_baru.groupby(['aspect', 'sentiment']).size().unstack(fill_value=0)

# Plot
sentimen_count.plot(kind='bar', stacked=True, figsize=(10, 6))
plt.title('Distribusi Sentimen per Aspek')
plt.xlabel('Aspek')
plt.ylabel('Jumlah')
plt.legend(title='Sentimen')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## **Visualisasi Aspek Per Sentimennya**

sentiment                           negative  neutral  positive
aspect                                                         
Dukungan terhadap Program                475      203      1119
Ketidaksetujuan dan Risiko Program      2695      877       807
Pelaksanaan Program di Sekolah          1833      297       744

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# Data
data = {
    'Aspek': [
        'Ketidaksetujuan dan\nRisiko Program',
        'Pelaksanaan Program\ndi Sekolah',
        'Dukungan terhadap\nProgram'
    ],
     'positif': [807, 744, 1119],
    'netral': [877, 297, 203],
    'negatif': [2695, 1833, 475]
}

df = pd.DataFrame(data)
df.set_index('Aspek', inplace=True)

# Ubah ke persen
df_percent = df.div(df.sum(axis=1), axis=0) * 100

# Plot
ax = df_percent.plot(
    kind='bar',
    stacked=True,
    color=['#38b000', '#fb6107', '#3772ff'] # Positif, Netral, Negatif
)

# Label persen
for i in range(len(df_percent)):
    cumulative = 0
    for j, col in enumerate(df_percent.columns):
        value = df_percent.iloc[i, j]
        if value > 5:
            ax.text(i, cumulative + value/2, f"{value:.1f}%", ha='center', va='center')
        cumulative += value

plt.xticks(rotation=0)
plt.ylabel("Persentase (%)")
plt.title("Distribusi Sentimen per Aspek")
plt.legend(title="Sentimen", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.show()

# **Split Data**

Data Train: 70%

Data Validasi: 15%

Data Test: 15%

In [ ]:
mbg_baru.columns

In [ ]:
# Menggabungkan kolom teks normalisasi dan aspek
mbg_baru['teks_aspek'] = mbg_baru['normalized_text'] + " [SEP] " + mbg_baru['aspect']
mbg_baru['teks_aspek']

In [ ]:
# Menampilkan seluruh teks tanpa pemotongan
pd.set_option('display.max_colwidth', None)
mbg_baru['teks_aspek'].head(5)

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split

X = mbg_baru['teks_aspek']
y = mbg_baru['sentiment_int']
aspect = mbg_baru['aspect']

X_train, X_temp, y_train, y_temp, aspect_train, aspect_temp = train_test_split(
    X, y, aspect,
    test_size=0.3,
    stratify=y,
    random_state=42
)

X_val, X_test, y_val, y_test, aspect_val, aspect_test = train_test_split(
    X_temp, y_temp, aspect_temp,
    test_size=0.5,
    stratify=y_temp,
    random_state=42
)

print("="*40)
print("DATA SPLIT (ASPEK + SENTIMEN):")
print(f"Total data: {len(X)}")

print(f"Train: {len(X_train)} ({len(X_train)/len(X)*100:.1f}%)")
print(f"Validation: {len(X_val)} ({len(X_val)/len(X)*100:.1f}%)")
print(f"Test: {len(X_test)} ({len(X_test)/len(X)*100:.1f}%)")
print("="*40)

In [ ]:
X

In [ ]:
X_train

In [ ]:
X_test

In [ ]:
X_val

In [ ]:
# Menampilkan beberapa data train
print("X_train:")
print(X_train.head())

print("\ny_train:")
print(y_train.head())

print("\naspect_train:")
print(aspect_train.head())

In [ ]:
y

In [ ]:
y_train

In [ ]:
aspect_train.value_counts()

In [ ]:
aspect_val.value_counts()

In [ ]:
aspect_test.value_counts()

# **Balancing data menggunakan Class Weight**

In [ ]:
from sklearn.utils.class_weight import compute_class_weight
import numpy as np

classes = np.unique(y_train)

class_weights = compute_class_weight(
    class_weight='balanced',
    classes=classes,
    y=y_train
)

class_weight_dict = dict(zip(classes, class_weights))
class_weight_dict

In [ ]:
# Cek total data train
from collections import Counter
Counter(y_train)

# **Text Embedding Using IndoBERT**

In [ ]:
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer
from transformers import BertTokenizer, BertModel
from sklearn.preprocessing import LabelEncoder

In [ ]:
# Load IndoBERT
tokenizer  = BertTokenizer.from_pretrained('indobenchmark/indobert-base-p2')
bert_model = BertModel.from_pretrained('indobenchmark/indobert-base-p2')
bert_model.eval()

In [ ]:
num_classes = len(mbg_baru['sentiment_int'].unique())
num_classes

In [ ]:
from tqdm import tqdm

def get_embeddings_batched(texts, mode='sequence', batch_size=32):
    all_embeddings = []

    for i in tqdm(range(0, len(texts), batch_size), desc=f"Embedding ({mode})"):
        batch = texts[i : i + batch_size]

        inputs = tokenizer(
            batch,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            max_length=128
        )

        with torch.no_grad():
            outputs = bert_model(**inputs)

        if mode == 'sequence':
            emb = outputs.last_hidden_state           # (batch, 128, 768)
        elif mode == 'cls':
            emb = outputs.last_hidden_state[:, 0, :]  # (batch, 768)

        all_embeddings.append(emb)

    return torch.cat(all_embeddings, dim=0)

In [ ]:
# EMBEDDING UNTUK CNN & BiLSTM → mode='sequence'
# CNN  : butuh semua token untuk ditangkap pola lokal lewat kernel/filter
# BiLSTM: butuh urutan token untuk diproses satu per satu
# ============================================================
print("Embedding sequence untuk CNN & BiLSTM...")

seq_train = get_embeddings_batched(X_train.tolist(), mode='sequence', batch_size=32)  # (n_train, 128, 768)
seq_val   = get_embeddings_batched(X_val.tolist(),   mode='sequence', batch_size=32)  # (n_val, 128, 768)
seq_test  = get_embeddings_batched(X_test.tolist(),  mode='sequence', batch_size=32)  # (n_test, 128, 768)

print(f"seq_train : {seq_train.shape}")
print(f"seq_val   : {seq_val.shape}")
print(f"seq_test  : {seq_test.shape}")

In [ ]:
print(f"seq_train : {seq_train.shape}")
print(f"seq_val   : {seq_val.shape}")
print(f"seq_test  : {seq_test.shape}")

In [ ]:
# EMBEDDING UNTUK INDOBERT FINE-TUNE → mode='cls'
# BERT sudah merangkum seluruh kalimat di token [CLS]
# Classifier tinggal membaca vektor itu langsung
# ============================================================
print("\nEmbedding CLS untuk IndoBERT...")

cls_train = get_embeddings_batched(X_train.tolist(), mode='cls')  # (n_train, 768)
cls_val   = get_embeddings_batched(X_val.tolist(),   mode='cls')  # (n_val, 768)
cls_test  = get_embeddings_batched(X_test.tolist(),  mode='cls')  # (n_test, 768)

print(f"cls_train : {cls_train.shape}")
print(f"cls_val   : {cls_val.shape}")
print(f"cls_test  : {cls_test.shape}")

In [ ]:
print(f"cls_train : {cls_train.shape}")
print(f"cls_val   : {cls_val.shape}")
print(f"cls_test  : {cls_test.shape}")

In [ ]:
le = LabelEncoder()
le.fit(mbg_baru['sentiment_int'])

y_train_tensor = torch.tensor(le.transform(y_train), dtype=torch.long)
y_val_tensor   = torch.tensor(le.transform(y_val),   dtype=torch.long)
y_test_tensor  = torch.tensor(le.transform(y_test),  dtype=torch.long)

**Melihat Hasil Text Embedding**

In [ ]:
# Lihat hasil tokenizer
sample_text = X_train.iloc[0]

inputs = tokenizer(
    sample_text,
    return_tensors='pt',
    padding='max_length',
    truncation=True,
    max_length=128
)

print("Teks:", sample_text)
print("\nToken:")
print(tokenizer.convert_ids_to_tokens(inputs['input_ids'][0]))

print("\nInput IDs:")
print(inputs['input_ids'])

print("\nToken Type IDs (segment):")
print(inputs['token_type_ids'])

print("\nAttention Mask:")
print(inputs['attention_mask'])

In [ ]:
# Ambil token embedding (word embedding)

bert_model.embeddings.word_embeddings

word_embeddings = bert_model.embeddings.word_embeddings(inputs['input_ids'])

print("Word Embedding shape:", word_embeddings.shape)
# (1, max_length, 768)

print("\nContoh embedding token pertama:")
print(word_embeddings[0][0][:10])  # 10 dimensi pertama


In [ ]:
# Ambil position embedding
position_ids = torch.arange(inputs['input_ids'].size(1)).unsqueeze(0)

position_embeddings = bert_model.embeddings.position_embeddings(position_ids)
position_embeddings

In [ ]:
print("Position Embedding shape:", position_embeddings.shape)

In [ ]:
# Segment Embedding
segment_embeddings = bert_model.embeddings.token_type_embeddings(inputs['token_type_ids'])

print("Segment Embedding shape:", segment_embeddings.shape)

In [ ]:
# Gabungan embedding -> embedding = token + position + segment
final_embedding = word_embeddings + position_embeddings + segment_embeddings

print("Final Embedding shape:", final_embedding.shape)

In [ ]:
def tampilkan_embedding_dari_data(index=0, max_length=40):
    text = X_train.iloc[index]

    inputs = tokenizer(
        text,
        return_tensors='pt',
        padding='max_length',
        truncation=True,
        max_length=max_length
    )

    tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
    input_ids = inputs['input_ids'][0].tolist()
    token_type_ids = inputs['token_type_ids'][0].tolist()
    attention_mask = inputs['attention_mask'][0].tolist()
    position_ids = list(range(1, len(input_ids)+1))

    print("=== ORIGINAL ===")
    print(text)

    print("\n=== TOKEN KHUSUS ===")
    print("[CLS] +", text, "+ [SEP]")

    print("\n=== JUMLAH TOKEN ===")
    print(len(tokens))

    print("\n=== WORD TOKENIZING ===")
    print(tokens)

    print("\n=== TOKEN EMBEDDING (input_ids) ===")
    print(input_ids)

    print("\n=== SEGMENT EMBEDDING ===")
    print(token_type_ids)

    print("\n=== POSITION EMBEDDING ===")
    print(position_ids)

    print("\n=== ATTENTION MASK ===")
    print(attention_mask)

In [ ]:
tampilkan_embedding_dari_data(index=0)

In [ ]:
tampilkan_embedding_dari_data(index=10)

In [ ]:
tampilkan_embedding_dari_data(index=3)

In [ ]:
def tampilkan_beberapa_embedding(n=15, max_length=40):
    for idx in range(n):
        text = X_train.iloc[idx]

        inputs = tokenizer(
            text,
            return_tensors='pt',
            padding='max_length',
            truncation=True,
            max_length=max_length
        )

        tokens = tokenizer.convert_ids_to_tokens(inputs['input_ids'][0])
        input_ids = inputs['input_ids'][0].tolist()
        token_type_ids = inputs['token_type_ids'][0].tolist()
        position_ids = list(range(1, len(input_ids)+1))

        print(f"\n================= DATA KE-{idx+1} =================")

        print("\nOriginal")
        print(text)

        print("\nToken Khusus")
        print("[CLS] +", text, "+ [SEP]")

        print("\nJumlah Token")
        print(len(tokens))

        print("\nWord Tokenizing")
        print(tokens)

        print("\nToken Embedding")
        print(input_ids)

        print("\nSegmen Embedding")
        print(token_type_ids)

        print("\nPosition Embedding")
        print(position_ids)

In [ ]:
tampilkan_beberapa_embedding(n=15)

# **ABSA**

## **Klasifikasi Sentimen**

##**CNN**

Epoch = 10

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import Input

# Konversi ke numpy
X_train_seq_cnn = seq_train.numpy()  # (n_train, 128, 768)
X_val_seq_cnn   = seq_val.numpy()    # (n_val, 128, 768)
X_test_seq_cnn  = seq_test.numpy()   # (n_test, 128, 768)

# Model CNN
num_classes = len(np.unique(y_train))

model_cnn = Sequential([
    Input(shape=(128, 768)),
    Conv1D(filters=128, kernel_size=5, activation='relu'),
    MaxPooling1D(pool_size=2),
    Flatten(),
    Dense(64, activation='relu'),
    Dropout(0.5),
    Dense(num_classes, activation='softmax')
])

model_cnn.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_model_cnn = model_cnn.fit(
    X_train_seq_cnn, y_train,
    validation_data=(X_val_seq_cnn, y_val),
    epochs=10,
    batch_size=32,
    class_weight=class_weight_dict,
    verbose=1
)


In [ ]:
# Classification Report Aspek Menyeluruh Data Test
from sklearn.metrics import classification_report
import numpy as np

y_pred_cnn_sent_all = np.argmax(
    model_cnn.predict(X_test_seq_cnn),
    axis=1
)

print(classification_report(
    y_test,
    y_pred_cnn_sent_all,
    digits=4
))

In [ ]:
# Prediksi
y_pred_cnn = np.argmax(model_cnn.predict(X_test_seq_cnn), axis=1)

In [ ]:
from sklearn.metrics import confusion_matrix

cm_cnn_sent = confusion_matrix(y_test, y_pred_cnn)
print(cm_cnn_sent)

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Define sentiment names for the plot
sentiment_names = [
    'Positif',
    'Netral',
    'Negatif'
]

cm_aspect_cnn = confusion_matrix(
    y_test,
    y_pred_cnn
)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_cnn_sent,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=sentiment_names,
    yticklabels=sentiment_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – CNN Sentimen Classification")
plt.tight_layout()
plt.show()

In [ ]:
# Plot Loss (rentang dinamis tapi minimal dari 0)
plt.figure(figsize=(6, 4))
plt.plot(history_model_cnn.history['loss'], label='Train Loss')
plt.plot(history_model_cnn.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss (CNN Sentiment)')
plt.legend()
plt.ylim(bottom=0)  # ← batas bawah 0, batas atas otomatis
plt.show()

# Plot Akurasi (rentang 0-1)
plt.figure(figsize=(6, 4))
plt.plot(history_model_cnn.history['accuracy'], label='Train Accuracy')
plt.plot(history_model_cnn.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy (CNN Sentiment)')
plt.legend()
plt.ylim(0, 1.0)  # ← akurasi pasti antara 0-1
plt.show()

In [ ]:
# Plot Loss dan Akurasi Bersebelahan (1 gambar, 2 subplot)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Subplot 1: Loss
ax1.plot(history_model_cnn.history['loss'], label='Train Loss', color='blue')
ax1.plot(history_model_cnn.history['val_loss'], label='Validation Loss', color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss (CNN Sentiment)')
ax1.set_ylim(0, 1.0)  # rentang Y 0-1
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.7)

# Subplot 2: Akurasi
ax2.plot(history_model_cnn.history['accuracy'], label='Train Accuracy', color='blue')
ax2.plot(history_model_cnn.history['val_accuracy'], label='Validation Accuracy', color='red')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy (CNN Sentiment)')
ax2.set_ylim(0, 1.0)  # rentang Y 0-1
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

# Layout rapi
plt.tight_layout()
plt.show()

**Evaluasi Model untuk Data Train, Validasi, dan Testing**

In [ ]:
# Evaluasi Data Train CNN-Aspek
y_pred_train_cnn = np.argmax(
    model_cnn.predict(X_train_seq_cnn),
    axis=1
)

print("Classification Report - Train")
print(classification_report(
    y_train,
    y_pred_train_cnn,
    digits=4
))

In [ ]:
# Evaluasi Data Validasi CNN-Aspek
y_pred_val_cnn = np.argmax(
    model_cnn.predict(X_val_seq_cnn),
    axis=1
)

print("Classification Report - Validation")
print(classification_report(
    y_val,
    y_pred_val_cnn,
    digits=4
))

##**BiLSTM**

**Yang bawah ini normalnya, cuma dikecilin bagian ukuran layernya, ga overfitting di epoch ke-6**

**Model ini yang aku coba**

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Bidirectional, LSTM, Dense, Dropout
from tensorflow.keras.optimizers import Adam
import numpy as np

# Konversi ke numpy
X_train_seq_bilstm = seq_train.numpy()  # (n_train, 128, 768)
X_val_seq_bilstm   = seq_val.numpy()    # (n_val, 128, 768)
X_test_seq_bilstm  = seq_test.numpy()   # (n_test, 128, 768)

# Model BiLSTM
num_classes = len(np.unique(y_train))

model_bilstm = Sequential([
    Bidirectional(
        LSTM(128, return_sequences=False),
        input_shape=(128, 768)  # ← pindahkan input_shape ke sini
    ),
    Dropout(0.5),  # ← hanya 1 dropout, hapus yang satunya
    Dense(64, activation='relu'),
    Dense(num_classes, activation='softmax')
])

model_bilstm.compile(
    optimizer=Adam(learning_rate=1e-4),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history_model_bilstm = model_bilstm.fit(
    X_train_seq_bilstm,
    y_train,
    validation_data=(X_val_seq_bilstm, y_val),
    epochs=10,
    batch_size=32,
    class_weight=class_weight_dict,
    verbose=1
)

In [ ]:
test_loss, test_acc = model_bilstm.evaluate(
    X_test_seq_bilstm,
    y_test,
    verbose=0
)

print("Test Accuracy (BiLSTM – Sentimen):", test_acc)


In [ ]:
import numpy as np

y_pred_bilstm = np.argmax(
   model_bilstm.predict(X_test_seq_bilstm),
    axis=1
)


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(
    y_test,
    y_pred_bilstm,
    digits=4
))


In [ ]:
from sklearn.metrics import confusion_matrix

cm_bilstm = confusion_matrix(
    y_test,
    y_pred_bilstm
)
cm_bilstm

In [ ]:
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt

# Mapping nama sentimen
sentiment_names = [
    'Positif',
    'Netral',
    'Negatif'
]

cm_bilstm = confusion_matrix(
    y_test,
    y_pred_bilstm
)

plt.figure(figsize=(7, 6))
sns.heatmap(
    cm_bilstm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=sentiment_names,
    yticklabels=sentiment_names
)

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix – BiLSTM Sentiment Classification")
plt.tight_layout()
plt.show()

In [ ]:
# Plot Loss dan Akurasi
# Loss
plt.figure(figsize=(6, 4))
plt.plot(history_model_bilstm.history['loss'], label='Train Loss')
plt.plot(history_model_bilstm.history['val_loss'], label='Validation Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss (BiLSTM Sentiment)')
plt.legend()
plt.show()

# Accuracy
plt.figure(figsize=(6, 4))
plt.plot(history_model_bilstm.history['accuracy'], label='Train Accuracy')
plt.plot(history_model_bilstm.history['val_accuracy'], label='Validation Accuracy')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Training and Validation Accuracy (BiLSTM Sentiment)')
plt.legend()
plt.show()

In [ ]:
# Plot Loss dan Akurasi Bersebelahan (1 gambar, 2 subplot)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Subplot 1: Loss
ax1.plot(history_model_bilstm.history['loss'], label='Train Loss', color='blue')
ax1.plot(history_model_bilstm.history['val_loss'], label='Validation Loss', color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss (BiLSTM Sentiment)')
ax1.set_ylim(0, 1.0)  # rentang Y 0-1
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.7)

# Subplot 2: Akurasi
ax2.plot(history_model_bilstm.history['accuracy'], label='Train Accuracy', color='blue')
ax2.plot(history_model_bilstm.history['val_accuracy'], label='Validation Accuracy', color='red')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy (CNN Sentiment)')
ax2.set_ylim(0, 1.0)  # rentang Y 0-1
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

# Layout rapi
plt.tight_layout()
plt.show()

In [ ]:
# Evaluasi Data Train
import numpy as np

y_pred_bilstm_train = np.argmax(
   model_bilstm.predict(X_train_seq_bilstm),
    axis=1
)
from sklearn.metrics import classification_report

print(classification_report(
    y_train,
    y_pred_bilstm_train,
    digits=4
))

In [ ]:
# Evaluasi Data Validasi
import numpy as np

y_pred_bilstm_val = np.argmax(
   model_bilstm.predict(X_val_seq_bilstm),
    axis=1
)
from sklearn.metrics import classification_report

print(classification_report(
    y_val,
    y_pred_bilstm_val,
    digits=4
))

# **Klasifikasi Berdasarkan Aspek**

## **CNN**

In [ ]:
# Prediksi
y_pred_cnn = np.argmax(model_cnn.predict(X_test_seq_cnn), axis=1)

In [ ]:
# =============================
# EVALUASI PER ASPEK
# =============================
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# Gabungkan jadi dataframe biar rapi
df_eval_cnn = pd.DataFrame({
    'aspect': aspect_test.values,
    'y_true': y_test.values,
    'y_pred': y_pred_cnn
})

# Loop tiap aspek
for asp in df_eval_cnn['aspect'].unique():
    print(f"\n===== ASPEK: {asp} =====")
    subset = df_eval_cnn[df_eval_cnn['aspect'] == asp]

    print("Jumlah data:", len(subset))
    print("Accuracy:", accuracy_score(subset['y_true'], subset['y_pred']))
    print(classification_report(subset['y_true'], subset['y_pred'], digits=4))

In [ ]:
labels = sorted(df_eval_cnn['y_true'].unique())

for asp in df_eval_cnn['aspect'].unique():
    print(f"\n===== ASPEK: {asp} =====")
    subset = df_eval_cnn[df_eval_cnn['aspect'] == asp]

    cm = confusion_matrix(subset['y_true'], subset['y_pred'], labels=labels)

    cm_df = pd.DataFrame(cm, index=labels, columns=labels)
    print(cm_df)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# mapping label
label_map = {0: 'Positif', 1: 'Netral', 2: 'Negatif'}

labels = sorted(df_eval_cnn['y_true'].unique())
label_names = [label_map[l] for l in labels]

for asp in df_eval_cnn['aspect'].unique():
    subset = df_eval_cnn[df_eval_cnn['aspect'] == asp]
    cm = confusion_matrix(subset['y_true'], subset['y_pred'], labels=labels)

    plt.figure(figsize=(5,4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=label_names,
        yticklabels=label_names
    )

    plt.title(f'CNN Aspek : {asp}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

## **BiLSTM**

In [ ]:
# Prediksi
y_pred_bilstm = np.argmax(model_bilstm.predict(X_test_seq_bilstm), axis=1)

In [ ]:
# EVALUASI PER ASPEK
# =============================
from sklearn.metrics import classification_report, accuracy_score
import pandas as pd

# Gabungkan jadi dataframe biar rapi
df_eval_bilstm = pd.DataFrame({
    'aspect': aspect_test.values,
    'y_true': y_test.values,
    'y_pred': y_pred_bilstm
})

# Loop tiap aspek
for asp in df_eval_bilstm['aspect'].unique():
    print(f"\n===== ASPEK: {asp} =====")
    subset = df_eval_bilstm[df_eval_bilstm['aspect'] == asp]

    print("Jumlah data:", len(subset))
    print("Accuracy:", accuracy_score(subset['y_true'], subset['y_pred']))
    print(classification_report(subset['y_true'], subset['y_pred'], digits=4))

In [ ]:
labels = sorted(df_eval_bilstm['y_true'].unique())

for asp in df_eval_bilstm['aspect'].unique():
    print(f"\n===== ASPEK: {asp} =====")
    subset = df_eval_bilstm[df_eval_bilstm['aspect'] == asp]

    cm = confusion_matrix(subset['y_true'], subset['y_pred'], labels=labels)

    cm_df_bilstm = pd.DataFrame(cm, index=labels, columns=labels)
    print(cm_df_bilstm)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# mapping label
label_map = {0: 'Positif', 1: 'Netral', 2: 'Negatif'}

labels = sorted(df_eval_bilstm['y_true'].unique())
label_names = [label_map[l] for l in labels]

for asp in df_eval_bilstm['aspect'].unique():
    subset = df_eval_bilstm[df_eval_bilstm['aspect'] == asp]
    cm = confusion_matrix(subset['y_true'], subset['y_pred'], labels=labels)

    plt.figure(figsize=(5,4))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=label_names,
        yticklabels=label_names
    )

    plt.title(f'BiLSTM - Aspek: {asp}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

 # **INDOBERT TERBARU**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from torch.optim import AdamW
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm import tqdm
import numpy as np

# 1. Definisi Model IndoBERT generik untuk klasifikasi
class IndoBERTClassifier(nn.Module):
    def __init__(self, model_name="indobenchmark/indobert-base-p2", num_classes=5, dropout=0.3):
        super(IndoBERTClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Mengambil pooled_output untuk klasifikasi (representasi CLS token)
        pooled_output = outputs.pooler_output
        dropped = self.dropout(pooled_output)
        logits = self.classifier(dropped)
        return logits

# 2. Dataset Class generik (Consolidated and fixed)
class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=128):
        # Explicitly convert to list to avoid Pandas Series indexing issues
        self.texts = list(texts)  # This will take values if 'texts' is a Series
        self.labels = list(labels)  # This will take values if 'labels' is a Series
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        text = str(self.texts[idx])
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_length,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long),
            'text': text  # Store original text for debugging/display
        }

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

def train_bert_model(model, train_loader, val_loader, class_weights_dict, epochs=5, lr=2e-6, model_task_name="Model"):

    # OPTIMIZER: Adam
    optimizer = optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)

    # LOSS FUNCTION
    if class_weights_dict:
        sorted_weights = [class_weights_dict[k] for k in sorted(class_weights_dict.keys())]
        class_weights_tensor = torch.tensor(sorted_weights, dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    else:
        criterion = nn.CrossEntropyLoss()

    best_val_f1 = 0
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
        'val_f1': [],
        'train_f1': []
    }

    for epoch in range(epochs):
        # ===================== TRAINING =====================
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []

        for batch in tqdm(train_loader, desc=f"{model_task_name} Epoch {epoch+1}/{epochs} [Train]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        # METRIK TRAIN
        train_acc = accuracy_score(train_labels, train_preds)
        train_f1 = f1_score(train_labels, train_preds, average='weighted')
        avg_train_loss = train_loss / len(train_loader)

        # ===================== VALIDATION =====================
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"{model_task_name} Epoch {epoch+1}/{epochs} [Val]"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                val_loss += loss.item()

                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # METRIK VALIDATION
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='weighted')
        avg_val_loss = val_loss / len(val_loader)

        # SIMPAN HISTORY
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        # PRINT
        print(f"\n{model_task_name} - Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"Val   Loss: {avg_val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

        # SAVE MODEL TERBAIK
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), f"best_{model_task_name.lower().replace(' ', '_')}.pth")
            print(f"✓ Model terbaik disimpan (F1: {val_f1:.4f})")

    return model, history

In [ ]:
# 4. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Load Tokenizer (gunakan tokenizer yang sudah didefinisikan sebelumnya)
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p2") # Sudah didefinisikan di awal

In [ ]:
# 6. Hitung jumlah kelas aspek
num_aspect_classes = len(np.unique(y_train_aspect))
print(f"Number of aspect classes: {num_aspect_classes}")

In [ ]:
# 7. Inisialisasi Model Aspek
model_aspect = IndoBERTClassifier(num_classes=num_aspect_classes).to(device)
model_aspect.float() # Ensure model parameters are float32

In [ ]:
# BUAT DATASET & DATALOADER dengan DATA ASLI (TANPA DIHAPUS!)
batch_size = 32

train_dataset_aspect = BERTDataset(X_train_aspect, y_train_aspect, tokenizer)
val_dataset_aspect = BERTDataset(X_val_aspect, y_val_aspect, tokenizer)
test_dataset_aspect = BERTDataset(X_test_aspect, y_test_aspect, tokenizer)

train_loader_aspect = DataLoader(train_dataset_aspect, batch_size=batch_size, shuffle=True)
val_loader_aspect = DataLoader(val_dataset_aspect, batch_size=batch_size, shuffle=False)
test_loader_aspect = DataLoader(test_dataset_aspect, batch_size=batch_size, shuffle=False)

In [ ]:
# Melihat ukuran dataset
print(f"Train dataset size: {len(train_dataset_aspect)}")
print(f"Val dataset size: {len(val_dataset_aspect)}")
print(f"Test dataset size: {len(test_dataset_aspect)}")

In [ ]:
# Melihat sampel pertama dari dataset
# Ambil sample pertama dari train dataset
sample = train_dataset_aspect[0]
print(f"Input IDs shape: {sample['input_ids'].shape}")
print(f"Attention mask shape: {sample['attention_mask'].shape}")
print(f"Label: {sample['labels']}")
print(f"Input IDs (first 10 tokens): {sample['input_ids'][:10]}")
print(f"Attention mask (first 10): {sample['attention_mask'][:10]}")


In [ ]:
# Melihat batch pertama dari dataloader
# Ambil satu batch dari train_loader
batch = next(iter(train_loader_aspect))
print(f"Batch input_ids shape: {batch['input_ids'].shape}")
print(f"Batch attention_mask shape: {batch['attention_mask'].shape}")
print(f"Batch labels shape: {batch['labels'].shape}")
print(f"Labels dalam batch: {batch['labels']}")

In [ ]:
#  Melihat distribusi label

from collections import Counter

# Untuk train
train_labels = [train_dataset_aspect[i]['labels'] for i in range(len(train_dataset_aspect))]
print(f"Train label distribution: {Counter(train_labels)}")

# Untuk val
val_labels = [val_dataset_aspect[i]['labels'] for i in range(len(val_dataset_aspect))]
print(f"Val label distribution: {Counter(val_labels)}")

# Untuk test
test_labels = [test_dataset_aspect[i]['labels'] for i in range(len(test_dataset_aspect))]
print(f"Test label distribution: {Counter(test_labels)}")

In [ ]:
class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        # Ensure texts and labels are converted to lists for positional indexing
        self.texts = texts.tolist() if hasattr(texts, 'tolist') else list(texts)
        self.labels = labels.tolist() if hasattr(labels, 'tolist') else list(labels)
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __getitem__(self, idx):
        text = str(self.texts[idx])  # This will now work with positional index
        label = self.labels[idx]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding='max_length',
            max_length=self.max_len,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].flatten(),
            'attention_mask': encoding['attention_mask'].flatten(),
            'labels': torch.tensor(label, dtype=torch.long),
            'text': text  # Store original text for debugging/display
        }

    def __len__(self):
        return len(self.labels)

# Re-instantiate train_dataset_aspect after the updated class definition
train_dataset_aspect = BERTDataset(X_train_aspect, y_train_aspect, tokenizer)

sample = train_dataset_aspect[0]
print(f"Original text: {sample['text']}")
print(f"Label: {sample['labels']}")
print(f"Input id: {sample['input_ids']}")
print(f"attention mask: {sample['attention_mask']}")

In [ ]:
for i in range(5):
    sample = train_dataset_aspect[i]

    print(f"=== DATA KE-{i+1} ===")
    print(f"Text           : {sample['text']}")
    print(f"Label          : {sample['labels']}")
    print(f"Input IDs      : {sample['input_ids']}")
    print(f"Attention Mask : {sample['attention_mask']}")
    print("-"*50)

In [ ]:
# Visualisasi panjang sequence
import matplotlib.pyplot as plt

def plot_sequence_lengths(dataset, title):
    lengths = []
    for i in range(min(1000, len(dataset))):  # Ambil 1000 sample
        input_ids = dataset[i]['input_ids']
        # Hitung panjang sebenarnya (tanpa padding)
        real_length = (input_ids != 0).sum().item()
        lengths.append(real_length)

    plt.figure(figsize=(10, 4))
    plt.hist(lengths, bins=50)
    plt.title(f'Sequence Length Distribution - {title}')
    plt.xlabel('Length')
    plt.ylabel('Frequency')
    plt.show()

    print(f"Mean length: {np.mean(lengths):.2f}")
    print(f"Max length: {max(lengths)}")
    print(f"Min length: {min(lengths)}")

plot_sequence_lengths(train_dataset_aspect, "Train")

In [ ]:
# INISIALISASI MODEL
model_aspect_indobert = IndoBERTClassifier(
    model_name="indobenchmark/indobert-base-p2",
    num_classes=num_aspect_classes,
    dropout=0.3
).to(device)
model_aspect_indobert.float()

# TRAINING
print("="*50)
print("TRAINING INDOBERT ASPECT CLASSIFICATION MODEL")
print("="*50)

model_aspect_indobert, history_aspect_indobert = train_bert_model(
    model_aspect_indobert,
    train_loader_aspect,
    val_loader_aspect,
    class_weights_dict=class_weight_dict_aspect,
    epochs=5,
    lr=2e-6,
    model_task_name="Aspect Classification IndoBERT"
)

In [ ]:
model_aspect_indobert.load_state_dict(
    torch.load("best_aspect_classification_indobert.pth")
)
model_aspect_indobert.eval()

In [ ]:
torch.save(
    model_aspect_indobert.state_dict(),
    "/content/drive/MyDrive/Tugas Akhir/Data/best_aspect_classification_indobert.pth"
)

In [ ]:
model_aspect_indobert.load_state_dict(
    torch.load("/content/drive/MyDrive/Tugas Akhir/Data/best_aspect_classification_indobert.pth")
)
model_aspect_indobert.eval()

In [ ]:
# Evaluasi Model Data Test
from sklearn.metrics import accuracy_score, f1_score

model_aspect_indobert.eval()
all_preds = []
all_labels = []

with torch.no_grad():
    for batch in test_loader_aspect:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model_aspect_indobert(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

# METRIK
acc = accuracy_score(all_labels, all_preds)
f1_weighted = f1_score(all_labels, all_preds, average='weighted')
f1_macro = f1_score(all_labels, all_preds, average='macro')

print("=== EVALUASI KESELURUHAN ===")
print(f"Accuracy       : {acc:.4f}")
print(f"F1-Weighted    : {f1_weighted:.4f}")
print(f"F1-Macro       : {f1_macro:.4f}")


In [ ]:
# Evaluasi Per Aspek
from sklearn.metrics import classification_report

# Mapping nama aspek
aspect_names = {
   0: 'Ketidaksetujuan & Risiko',
   1: 'Pelaksanaan di Sekolah',
   2: 'Dukungan Program'
}

aspect_categories = [aspect_names[i] for i in sorted(aspect_names.keys())]

print("\n=== CLASSIFICATION REPORT PER ASPEK ===")
print(classification_report(
    all_labels,
    all_preds,
    target_names=aspect_categories,
    digits=4
))

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Hitung confusion matrix
cm = confusion_matrix(all_labels, all_preds)

# Ubah ke DataFrame biar ada nama aspek
cm_df = pd.DataFrame(cm, index=aspect_categories, columns=aspect_categories)

# Plot
plt.figure(figsize=(7, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Aspect Classification IndoBERT")

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history_aspect_indobert['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 🔸 Subplot 1: Loss
ax1.plot(epochs, history_aspect_indobert['train_loss'],
         label='Train Loss', color='blue')
ax1.plot(epochs, history_aspect_indobert['val_loss'],
         label='Validation Loss', color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss (IndoBERT Aspect)')
ax1.set_ylim(0, 1.0)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.7)

# 🔸 Subplot 2: Accuracy
ax2.plot(epochs, history_aspect_indobert['train_acc'],
         label='Train Accuracy', color='blue')
ax2.plot(epochs, history_aspect_indobert['val_acc'],
         label='Validation Accuracy', color='red')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy (IndoBERT Aspect)')
ax2.set_ylim(0, 1.0)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
# Evaluasi Data Train dan Validasi

from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate_model(model, data_loader, device, aspect_categories, dataset_name="Data"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"\n=== EVALUASI {dataset_name.upper()} ===")

    # Classification report
    print("\nClassification Report:")
    print(classification_report(
        all_labels,
        all_preds,
        target_names=aspect_categories,
        digits=4
    ))

    return all_labels, all_preds


# Train
train_labels, train_preds = evaluate_model(
    model_aspect_indobert,
    train_loader_aspect,
    device,
    aspect_categories,
    dataset_name="Train"
)

# Validation
val_labels, val_preds = evaluate_model(
    model_aspect_indobert,
    val_loader_aspect,
    device,
    aspect_categories,
    dataset_name="Validation"
)

## **Klasfikasi Sentimen**

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.metrics import classification_report, accuracy_score, f1_score
from tqdm import tqdm
import numpy as np

# 1. Definisi Model IndoBERT generik untuk klasifikasi
class IndoBERTClassifier(nn.Module):
    def __init__(self, model_name="indobenchmark/indobert-base-p2", num_classes=5, dropout=0.3):
        super(IndoBERTClassifier, self).__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(dropout)
        self.classifier = nn.Linear(self.bert.config.hidden_size, num_classes)

    def forward(self, input_ids, attention_mask):
        outputs = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        # Mengambil pooled_output untuk klasifikasi (representasi CLS token)
        pooled_output = outputs.pooler_output
        dropped = self.dropout(pooled_output)
        logits = self.classifier(dropped)
        return logits

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score

def train_bert_model(model, train_loader, val_loader, class_weights_dict, epochs=5, lr=2e-6, model_task_name="Model"):

    # OPTIMIZER: AdamW
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-3)

    # LOSS FUNCTION
    if class_weights_dict:
        sorted_weights = [class_weights_dict[k] for k in sorted(class_weights_dict.keys())]
        class_weights_tensor = torch.tensor(sorted_weights, dtype=torch.float32).to(device)
        criterion = nn.CrossEntropyLoss(weight=class_weights_tensor)
    else:
        criterion = nn.CrossEntropyLoss()

    best_val_f1 = 0
    history = {
        'train_loss': [],
        'val_loss': [],
        'train_acc': [],
        'val_acc': [],
        'val_f1': [],
        'train_f1': []
    }

    for epoch in range(epochs):
        # ===================== TRAINING =====================
        model.train()
        train_loss = 0
        train_preds = []
        train_labels = []

        for batch in tqdm(train_loader, desc=f"{model_task_name} Epoch {epoch+1}/{epochs} [Train]"):
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            optimizer.zero_grad()
            logits = model(input_ids, attention_mask)
            loss = criterion(logits, labels)
            loss.backward()

            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            train_loss += loss.item()

            preds = torch.argmax(logits, dim=1)
            train_preds.extend(preds.cpu().numpy())
            train_labels.extend(labels.cpu().numpy())

        # METRIK TRAIN
        train_acc = accuracy_score(train_labels, train_preds)
        train_f1 = f1_score(train_labels, train_preds, average='weighted')
        avg_train_loss = train_loss / len(train_loader)

        # ===================== VALIDATION =====================
        model.eval()
        val_loss = 0
        all_preds = []
        all_labels = []

        with torch.no_grad():
            for batch in tqdm(val_loader, desc=f"{model_task_name} Epoch {epoch+1}/{epochs} [Val]"):
                input_ids = batch['input_ids'].to(device)
                attention_mask = batch['attention_mask'].to(device)
                labels = batch['labels'].to(device)

                logits = model(input_ids, attention_mask)
                loss = criterion(logits, labels)
                val_loss += loss.item()

                preds = torch.argmax(logits, dim=1)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

        # METRIK VALIDATION
        val_acc = accuracy_score(all_labels, all_preds)
        val_f1 = f1_score(all_labels, all_preds, average='weighted')
        avg_val_loss = val_loss / len(val_loader)

        # SIMPAN HISTORY
        history['train_loss'].append(avg_train_loss)
        history['val_loss'].append(avg_val_loss)
        history['train_acc'].append(train_acc)
        history['train_f1'].append(train_f1)
        history['val_acc'].append(val_acc)
        history['val_f1'].append(val_f1)

        # PRINT
        print(f"\n{model_task_name} - Epoch {epoch+1}/{epochs}")
        print(f"Train Loss: {avg_train_loss:.4f} | Acc: {train_acc:.4f} | F1: {train_f1:.4f}")
        print(f"Val   Loss: {avg_val_loss:.4f} | Acc: {val_acc:.4f} | F1: {val_f1:.4f}")

        # SAVE MODEL TERBAIK
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), f"best_{model_task_name.lower().replace(' ', '_')}.pth")
            print(f"✓ Model terbaik disimpan (F1: {val_f1:.4f})")

    return model, history

In [ ]:
# Hitung jumlah kelas aspek
num_classes = len(np.unique(y_train))
print(f"Number of sentiment classes: {num_classes}")

In [ ]:
# 4. Setup Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

In [ ]:
# Inisialisasi Model Aspek
model_sent_indo = IndoBERTClassifier(num_classes=num_classes).to(device)
model_sent_indo.float() # Ensure model parameters are float32

In [ ]:
# Load Tokenizer (gunakan tokenizer yang sudah didefinisikan sebelumnya)
tokenizer = AutoTokenizer.from_pretrained("indobenchmark/indobert-base-p2") # Sudah didefinisikan di awal

In [ ]:
class BERTDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_len=128):
        self.texts = texts
        self.labels = labels
        self.tokenizer = tokenizer
        self.max_len = max_len

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):

        text = str(self.texts.iloc[idx])
        label = self.labels.iloc[idx]

        encoding = self.tokenizer(
            text,
            add_special_tokens=True,
            max_length=self.max_len,
            padding='max_length',
            truncation=True,
            return_attention_mask=True,
            return_tensors='pt'
        )

        return {
            'input_ids': encoding['input_ids'].squeeze(0),
            'attention_mask': encoding['attention_mask'].squeeze(0),
            'labels': torch.tensor(label, dtype=torch.long)
        }

In [ ]:
# BUAT DATASET & DATALOADER dengan DATA ASLI
batch_size = 32

train_dataset_sentiment = BERTDataset(X_train, y_train, tokenizer)
val_dataset_sentiment = BERTDataset(X_val, y_val, tokenizer)
test_dataset_sentiment  = BERTDataset(X_test , y_test, tokenizer)

train_loader_sentiment  = DataLoader(train_dataset_sentiment, batch_size=batch_size, shuffle=True)
val_loader_sentiment  = DataLoader(val_dataset_sentiment, batch_size=batch_size, shuffle=False)
test_loader_sentiment  = DataLoader(test_dataset_sentiment, batch_size=batch_size, shuffle=False)

In [ ]:
print(f"Train sentiment dataset size: {len(train_dataset_sentiment)}")
print(f"Val sentiment dataset size: {len(val_dataset_sentiment)}")
print(f"Test sentiment dataset size: {len(test_dataset_sentiment)}")

In [ ]:
 # INISIALISASI MODEL
model_sentiment_indobert = IndoBERTClassifier(
    model_name="indobenchmark/indobert-base-p2",
    num_classes=num_classes,
    dropout=0.3
).to(device)
model_sentiment_indobert.float()

# TRAINING
print("="*50)
print("TRAINING INDOBERT SENTIMENT CLASSIFICATION MODEL")
print("="*50)

model_sentiment_indobert, history_sentiment_indobert = train_bert_model(
    model_sentiment_indobert,
    train_loader_sentiment,
    val_loader_sentiment,
    class_weights_dict=class_weight_dict,
    epochs=5,
    lr=2e-6,
    model_task_name="Sentiment Classification"
)

In [ ]:
model_sentiment_indobert.load_state_dict(
    torch.load("best_sentiment_classification.pth")
)
model_sentiment_indobert.eval()

In [ ]:
torch.save(
    model_sentiment_indobert.state_dict(),
    "/content/drive/MyDrive/Tugas Akhir/Data/best_sentiment_classification.pth"
)

In [ ]:
model_sentiment_indobert.load_state_dict(
    torch.load("/content/drive/MyDrive/Tugas Akhir/Data/best_sentiment_classification.pth")
)
model_sentiment_indobert.eval()

In [ ]:
# Evaluasi Model Data Test
from sklearn.metrics import accuracy_score, f1_score

model_sentiment_indobert.eval()
all_preds_sentiment = []
all_labels_sentiment = []

with torch.no_grad():
    for batch in test_loader_sentiment:
        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model_sentiment_indobert(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        all_preds_sentiment.extend(preds.cpu().numpy())
        all_labels_sentiment.extend(labels.cpu().numpy())

# METRIK
acc_sentiment = accuracy_score(all_labels_sentiment, all_preds_sentiment)
f1_weighted_sentiment = f1_score(all_labels_sentiment, all_preds_sentiment, average='weighted')
f1_macro_sentiment = f1_score(all_labels_sentiment, all_preds_sentiment, average='macro')

print("=== EVALUASI KESELURUHAN SENTIMEN ===")
print(f"Accuracy       : {acc_sentiment:.4f}")
print(f"F1-Weighted    : {f1_weighted_sentiment:.4f}")
print(f"F1-Macro       : {f1_macro_sentiment:.4f}")

In [ ]:
# Evaluasi Per Sentimen
from sklearn.metrics import classification_report

# Mapping nama sentimen
sentiment_names = {
   0: 'Positif',
   1: 'Netral',
   2: 'Negatif'
}

sentiment_categories = [sentiment_names[i] for i in sorted(sentiment_names.keys())]

print("\n=== CLASSIFICATION REPORT PER SENTIMEN ===")
print(classification_report(
    all_labels_sentiment,
    all_preds_sentiment,
    target_names=sentiment_categories,
    digits=4
))

In [ ]:
# Confusion Matrix
from sklearn.metrics import confusion_matrix
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd

# Hitung confusion matrix
cm = confusion_matrix(all_labels_sentiment, all_preds_sentiment)

# Mapping nama sentimen
sentiment_names = {
   0: 'Positif',
   1: 'Netral',
   2: 'Negatif'
}
sentiment_categories = [sentiment_names[i] for i in sorted(sentiment_names.keys())]

# Ubah ke DataFrame biar ada nama sentimen
cm_df = pd.DataFrame(cm, index=sentiment_categories, columns=sentiment_categories)

# Plot
plt.figure(figsize=(7, 6))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues')

plt.xlabel("Predicted Label")
plt.ylabel("True Label")
plt.title("Confusion Matrix - Sentiment Classification IndoBERT")

plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)

plt.tight_layout()
plt.show()

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history_sentiment_indobert['train_loss']) + 1)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# 🔸 Subplot 1: Loss
ax1.plot(epochs, history_sentiment_indobert['train_loss'],
         label='Train Loss', color='blue')
ax1.plot(epochs, history_sentiment_indobert['val_loss'],
         label='Validation Loss', color='red')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss (IndoBERT Sentiment)')
ax1.set_ylim(0, 1.0)
ax1.legend()
ax1.grid(True, linestyle='--', alpha=0.7)

# 🔸 Subplot 2: Accuracy
ax2.plot(epochs, history_sentiment_indobert['train_acc'],
         label='Train Accuracy', color='blue')
ax2.plot(epochs, history_sentiment_indobert['val_acc'],
         label='Validation Accuracy', color='red')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy')
ax2.set_title('Training and Validation Accuracy (IndoBERT Sentiment)')
ax2.set_ylim(0, 1.0)
ax2.legend()
ax2.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.show()

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np

def evaluate_model(model, data_loader, device, aspect_categories, dataset_name="Data"):
    model.eval()
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for batch in data_loader:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            labels = batch['labels'].to(device)

            logits = model(input_ids, attention_mask)
            preds = torch.argmax(logits, dim=1)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    print(f"\n=== EVALUASI {dataset_name.upper()} ===")

    # Classification report
    print("\nClassification Report:")
    print(classification_report(
        all_labels,
        all_preds,
        target_names=sentiment_categories,
        digits=4
    ))

    return all_labels, all_preds

In [ ]:
# Train
train_labels, train_preds = evaluate_model(
    model_sentiment_indobert,
    train_loader_sentiment,
    device,
    sentiment_categories,
    dataset_name="Train"
)

# Validation
val_labels, val_preds = evaluate_model(
    model_sentiment_indobert,
    val_loader_sentiment,
    device,
    sentiment_categories,
    dataset_name="Validation"
)

## **Klasifikasi Berdasarkan Aspek**

In [ ]:
all_aspects = []
all_preds_sentiment = []
all_labels_sentiment = []

In [ ]:
with torch.no_grad():
    for i, batch in enumerate(test_loader_sentiment):

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        labels = batch['labels'].to(device)

        logits = model_sentiment_indobert(input_ids, attention_mask)
        preds = torch.argmax(logits, dim=1)

        all_preds_sentiment.extend(preds.cpu().numpy())
        all_labels_sentiment.extend(labels.cpu().numpy())

In [ ]:
import pandas as pd

df_eval_indobert = pd.DataFrame({
    'aspect': aspect_test.reset_index(drop=True),
    'y_true': all_labels_sentiment,
    'y_pred': all_preds_sentiment
})

In [ ]:
from sklearn.metrics import classification_report

for asp in df_eval_indobert['aspect'].unique():

    print(f"\n===== ASPEK: {asp} =====")

    subset = df_eval_indobert[
        df_eval_indobert['aspect'] == asp
    ]

    print("Jumlah data:", len(subset))

    print(classification_report(
        subset['y_true'],
        subset['y_pred']
    ))

In [ ]:
from sklearn.metrics import confusion_matrix
import pandas as pd

labels = sorted(df_eval_indobert['y_true'].unique())

for asp in df_eval_indobert['aspect'].unique():

    print(f"\n===== ASPEK: {asp} =====")

    subset = df_eval_indobert[
        df_eval_indobert['aspect'] == asp
    ]

    cm = confusion_matrix(
        subset['y_true'],
        subset['y_pred'],
        labels=labels
    )

    cm_df_indobert = pd.DataFrame(
        cm,
        index=labels,
        columns=labels
    )

    print(cm_df_indobert)

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import confusion_matrix

# mapping label
label_map = {
    0: 'Positif',
    1: 'Netral',
    2: 'Negatif'
}

labels = sorted(df_eval_indobert['y_true'].unique())
label_names = [label_map[l] for l in labels]

for asp in df_eval_indobert['aspect'].unique():

    subset = df_eval_indobert[
        df_eval_indobert['aspect'] == asp
    ]

    cm = confusion_matrix(
        subset['y_true'],
        subset['y_pred'],
        labels=labels
    )

    plt.figure(figsize=(5,4))

    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=label_names,
        yticklabels=label_names
    )

    plt.title(f'IndoBERT - Aspek: {asp}')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')

    plt.tight_layout()
    plt.show()

# **Implementasi Model Terbaik (IndoBERT) ke Aspect-Based Sentiment Classification**

Sumber Komentar untuk Data Baru : [link text](https://vt.tiktok.com/ZSHoyBcuW/)

In [ ]:
# Input Data Baru
new_texts = [
    "tutup mbg dialihkan kependidikan gratis",
    "mbg tuh menurut ku ngga efektif, karena asli di lapangan makanannya banyak yang tidak disukai para siswa. kadang banyak yg kebuang, dan kadang dibawa para guru pulang karena mubazir. tolong lah pak Prabowo buka mata hati karena bukan program BPK yg salah. tp pengolahan dilapangan kurang tepat 😭",
    "padahal dengan adanya mbg banyak membawa dampak positif contohnya bisa menciptakan lapangan pekerjaan",
    "Anggaran nya gk cuma ngurangin pendidikan tp semua nya kenaa.. kn effisiensi.",
    "MBG maju terus"

]
new_texts

In [ ]:
data_baru = {
    "text": new_texts,
    "aspect": [0, 0, 2, 0, 2],
    "sentiment": [2, 2, 0, 2, 0]
}

df_new = pd.DataFrame(data_baru)

In [ ]:
# =========================
# DATASET DATA BARU
# =========================
new_dataset = BERTDataset(
    texts=df_new['text'],
    labels=df_new['sentiment'],
    tokenizer=tokenizer,
    max_len=128
)

new_loader = DataLoader(
    new_dataset,
    batch_size=1,
    shuffle=False
)

In [ ]:
# =========================
# PREDIKSI DATA BARU
# =========================
model_sentiment_indobert.eval()

predictions = []

with torch.no_grad():

    for batch in new_loader:

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)

        logits = model_sentiment_indobert(
            input_ids,
            attention_mask
        )

        preds = torch.argmax(logits, dim=1)

        predictions.extend(
            preds.cpu().numpy()
        )

In [ ]:
df_result = df_new.copy()

df_result['pred_sentiment'] = predictions

In [ ]:
label_map_sentiment = {
    0: "Positif",
    1: "Netral",
    2: "Negatif"
}

label_map_aspect = {
    0: "Ketidaksetujuan dan Risiko Program",
    1: "Pelaksanaan Program di Sekolah",
    2: "Dampak Positif Program"
}

In [ ]:
df_result['aspect_name'] = df_result['aspect'].map(label_map_aspect)

df_result['sentiment'] = (
    df_result['sentiment']
    .map(label_map_sentiment)
)

df_result['pred_sentiment'] = (
    df_result['pred_sentiment']
    .map(label_map_sentiment)
)

In [ ]:
df_result[[
    'text',
    'aspect_name',
    'sentiment',
    'pred_sentiment'
]]